# CO2 Emissions Explorer - Data Visualization
This notebook contains the data processing and visualization code from the Streamlit dashboard exercise, adapted for interactive exploration in a Jupyter Notebook environment.

In [ ]:
import pandas as pd
import plotly.express as px
from pathlib import Path

# Load data (adjusting path for notebook location)
path = Path('../data/co2_emissions.csv')
df = pd.read_csv(path)
df['Date'] = pd.to_datetime(df['Year'].astype(str) + '-01-01')

df.head()

## Define Filters
Change the variables below to simulate the behavior of the Streamlit widgets. This allows you to explore different regions, countries, dates, and metrics.

In [ ]:
# ── Simulated Widgets ────────────────────────────────────────────────────────
selected_region = "All"
# Or select a specific region:
# selected_region = "North America"

if selected_region == "All":
    country_options = sorted(df["Country"].unique().tolist())
else:
    country_options = sorted(df[df["Region"] == selected_region]["Country"].unique().tolist())

# Select a few countries to compare
selected_countries = ["United States", "China", "India", "Germany"]

# Date range (using pandas Timestamps for filtering)
start_date = pd.to_datetime('2000-01-01')
end_date = pd.to_datetime('2022-01-01')

# Metric to visualize
# Options: "CO2_Mt" (Total CO2) or "CO2_per_capita"
selected_metric = "CO2_Mt" 
selected_metric_label = "Total CO2 (Mt)"

# Whether to highlight the top emitter
highlight_top = True

# ── Filtering Data ──────────────────────────────────────────────────────────
filtered = df[
    (df['Country'].isin(selected_countries)) &
    (df['Date'] >= start_date) &
    (df['Date'] <= end_date)
]

print(f"Data filtered: {len(selected_countries)} countries | {selected_region} | {start_date.year} - {end_date.year} | {selected_metric_label}")

## Key Performance Indicators (KPIs)

In [ ]:
if not filtered.empty:
    first_year_data = filtered[filtered['Date'] == filtered['Date'].min()]
    last_year_data = filtered[filtered['Date'] == filtered['Date'].max()]
    
    total_last = last_year_data['CO2_Mt'].sum()
    total_first = first_year_data['CO2_Mt'].sum()
    pct_change = ((total_last - total_first) / total_first * 100) if total_first else 0
    
    top_country = last_year_data.loc[last_year_data['CO2_Mt'].idxmax()]['Country'] if not last_year_data.empty else "N/A"
    
    print(f"Total CO2 (Mt) in last year ({end_date.year}): {total_last:,.1f}")
    print(f"Change from first year ({start_date.year}): {pct_change:+.1f}%")
    print(f"Top Emitter (last year): {top_country}")

## Visualizations

In [ ]:
# ── Line Chart: Trend over time ──────────────────────────────────────────────
if not filtered.empty:
    total_emissions = filtered.groupby('Country')[selected_metric].sum()
    top_emitter = total_emissions.idxmax() if not total_emissions.empty else None
else:
    top_emitter = None

if highlight_top and top_emitter:
    color_discrete_map = {c: 'lightgrey' for c in selected_countries}
    color_discrete_map[top_emitter] = '#d62728' # highlight in red
    
    fig_line = px.line(
        filtered, x='Date', y=selected_metric, color='Country',
        color_discrete_map=color_discrete_map,
        title=f"Trend of {selected_metric_label} over time"
    )
    
    end_data = filtered[filtered['Country'] == top_emitter]
    if not end_data.empty:
        end_point = end_data.iloc[-1]
        fig_line.add_annotation(
            x=end_point['Date'],
            y=end_point[selected_metric],
            text=top_emitter,
            showarrow=False,
            xanchor='left',
            xshift=5,
            font=dict(color='#d62728', size=12)
        )
    fig_line.update_layout(showlegend=False)
else:
    fig_line = px.line(
        filtered, x='Date', y=selected_metric, color='Country',
        title=f"Trend of {selected_metric_label} over time"
    )

fig_line.update_layout(
    plot_bgcolor='white',
    paper_bgcolor='white',
    xaxis=dict(showgrid=False),
    yaxis=dict(showgrid=True, gridcolor='lightgrey')
)

fig_line.show()

# ── Bar Chart: Ranking in last year ──────────────────────────────────────────
if not filtered.empty:
    last_year_data = filtered[filtered['Date'] == filtered['Date'].max()]
    bar_data = last_year_data.sort_values(selected_metric, ascending=True)
    
    fig_bar = px.bar(
        bar_data, x=selected_metric, y='Country', orientation='h',
        title=f"Ranking in {end_date.year}",
        color_discrete_sequence=['#1f77b4'],
        height=400 + (len(selected_countries) * 20) # dynamic height
    )
    
    fig_bar.update_layout(
        plot_bgcolor='white',
        paper_bgcolor='white',
        xaxis=dict(showgrid=True, gridcolor='lightgrey'),
        yaxis=dict(showgrid=False)
    )
    
    fig_bar.show()